# Evaluación RAGAS — Asistente de Consulta Jurídica Laboral

SI4006 · Equipo Lawten

---

### 1. Marco Teórico y Objetivos de RAGAS en M3

El harness de M2 evalúa el sistema de extremo a extremo mediante métricas de producto y un juez general. **RAGAS (Retrieval Augmented Generation Assessment)** complementa esa visión mediante un **diagnóstico desacoplado** de los dos componentes centrales del pipeline RAG:

```
                [ Consulta (Q) ]
                   /        \
                  /          \
       [ Contextos (C) ]  ->  [ Resumen (A) ]
                  \          /
                   \        /
              [ Referencia (R) ]
```

1. **Componente de Recuperación (Retriever):**
   * **`context_recall`**: Fracción de las normas y hechos clave de la verdad terreno (`expected`) presentes en los artículos recuperados. Si es bajo, la falla proviene directamente del buscador.
   * **`context_precision`**: Medida de ranking (Average Precision at k) que evalúa si los artículos relevantes quedaron en las primeras posiciones (top-1, top-2) y no enterrados tras normas irrelevantes.

2. **Componente de Generación (Decoder / Qwen2.5-1.5B):**
   * **`faithfulness`**: Fidelidad fáctica del texto generado respecto a los artículos recuperados. Verifica que el modelo no invente normas ni atributos ausentes de los textos.
   * **`answer_relevancy`**: Pertinencia de la respuesta generada respecto a la consulta original del trabajador.

#### Decisiones metodológicas:
* **Variable `respuesta`:** Se evalúa **exclusivamente el resumen** generado por el LLM (`Qwen2.5-1.5B-Instruct`), recortado entre `Resumen:` y el aviso legal. La lista de referencias numeradas y el descargo legal son construidos por código en Python y no representan generación del modelo.
* **Variable `contextos`:** Se utilizan los **textos completos** (`texto_completo`) del corpus `data/diccionario_articulos.csv` para cada cita de `articulos_recuperados`, y no las descripciones truncadas a 400 caracteres que ve el decoder.
* **Modelo de Juez:** `claude-haiku-4-5` vía la API de Anthropic (autorizado por el docente). El juez es un LLM y su puntaje varía entre corridas con las mismas entradas, por lo que las diferencias pequeñas entre sistemas (del orden de 0,1) no deben interpretarse como mejoras.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Bosnape/cabrejos-ortiz-valencia-lopez.git'
REPO_BRANCH = 'main'
NOTEBOOK_REL = Path('notebooks/rag/ragas_m3.ipynb')

try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def buscar_raiz(inicio):
    inicio = Path(inicio).resolve()
    return next((p for p in (inicio, *inicio.parents) if (p / NOTEBOOK_REL).is_file()), None)

RAIZ = buscar_raiz(Path.cwd())
if RAIZ is None and EN_COLAB:
    destino = Path('/content/cabrejos-ortiz-valencia-lopez')
    if not destino.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(destino)], check=True)
    RAIZ = destino

if RAIZ is None:
    RAIZ = Path('.').resolve()
os.chdir(RAIZ)

if EN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'anthropic==1.4.0', 'pandas==3.0.5', 'python-dotenv==1.2.3'], check=True)

requeridos = [
    Path('notebooks/evaluation/eval_set.json'),
    Path('data/diccionario_articulos.csv'),
    Path('notebooks/rag/resultados/ingenuo/scorecard_rag_ingenuo_detalle.csv'),
    Path('notebooks/rag/resultados/avanzado/scorecard_rag_avanzado_detalle.csv'),
]
faltantes = [str(r) for r in requeridos if not (RAIZ / r).is_file()]
if faltantes:
    raise FileNotFoundError('Faltan archivos requeridos: ' + ', '.join(faltantes))
print('Raíz del proyecto verificada:', RAIZ)

Raíz del proyecto verificada: /content/cabrejos-ortiz-valencia-lopez


In [2]:
import random
import numpy as np
from dotenv import load_dotenv
import anthropic

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

load_dotenv(RAIZ / '.env', override=False)
load_dotenv(RAIZ.parent / '.env', override=False)

if EN_COLAB and not os.getenv('ANTHROPIC_API_KEY'):
    try:
        from google.colab import userdata
        clave = userdata.get('ANTHROPIC_API_KEY')
        if clave:
            os.environ['ANTHROPIC_API_KEY'] = clave
    except Exception:
        pass

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
if not ANTHROPIC_API_KEY:
    raise RuntimeError('Configure ANTHROPIC_API_KEY en el archivo .env o en los Secretos de Colab.')

MODELO_JUEZ = 'claude-haiku-4-5'
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f'Juez LLM configurado: {MODELO_JUEZ}')

Juez LLM configurado: claude-haiku-4-5


In [3]:
import json
import re
import pandas as pd

with open(RAIZ / 'notebooks/evaluation/eval_set.json', 'r', encoding='utf-8') as f:
    eval_set = json.load(f)
eval_map = {item['id']: item for item in eval_set}

df_dic = pd.read_csv(RAIZ / 'data/diccionario_articulos.csv')
cita_to_texto = {}
for _, row in df_dic.iterrows():
    fuente = str(row['fuente']).strip()
    numero = str(row['numero']).strip()
    cita = f"{fuente} Art. {numero}"
    cita_to_texto[cita] = str(row['texto_completo'])

def extraer_resumen(respuesta_sistema):
    """Extrae exclusivamente el resumen generado por Qwen, descartando referencias y disclaimer."""
    if pd.isna(respuesta_sistema):
        return ''
    texto = str(respuesta_sistema)
    m = re.search(r'Resumen:\s*(.*?)(?=\r?\n\r?\nEste resultado es informativo|$)', texto, re.DOTALL)
    if m:
        return m.group(1).strip()
    return texto.strip()

def obtener_contextos(articulos_recuperados_str):
    """Obtiene los textos completos del diccionario para cada cita recuperada."""
    if pd.isna(articulos_recuperados_str):
        return []
    contextos = []
    for cita in str(articulos_recuperados_str).split('|'):
        cita_clean = cita.strip()
        if cita_clean in cita_to_texto:
            contextos.append(cita_to_texto[cita_clean])
        else:
            print(f"[WARN] Cita no encontrada en el diccionario: '{cita_clean}' — se omite del contexto.")
    return contextos

print(f'Corpus cargado: {len(cita_to_texto)} artículos indexados.')
print(f'Eval set cargado: {len(eval_set)} casos de prueba.')

Corpus cargado: 143 artículos indexados.
Eval set cargado: 13 casos de prueba.


In [4]:

def limpiar_json_salida(texto):
    texto = texto.strip()
    if texto.startswith(('```', '~~~')):
        m = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', texto)
        if m:
            texto = m.group(1).strip()
    return texto

def llamar_claude_json(prompt, max_tokens=1000):
    for intento in range(3):
        try:
            resp = client.messages.create(
                model=MODELO_JUEZ,
                max_tokens=max_tokens,
                messages=[{'role': 'user', 'content': prompt}]
            )
            raw = resp.content[0].text
            clean = limpiar_json_salida(raw)
            return json.loads(clean)
        except Exception as e:
            print(f"[WARN] llamar_claude_json intento {intento+1}/3 falló: {e}")
            if intento == 2:
                print("[ERROR] 3 intentos fallidos; se guardará NaN en lugar de 0.0.")
                return None

def calcular_faithfulness(respuesta, contextos):
    if not respuesta:
        return float('nan')
    contextos_str = "\n\n---\n\n".join(f"[Articulo {i+1}]: {c}" for i, c in enumerate(contextos))
    prompt = f"""Eres un evaluador riguroso de fidelidad fáctica para sistemas RAG en derecho colombiano.
Tu tarea es verificar si cada afirmación presente en el 'Resumen' está directamente respaldada por los 'Contextos' proporcionados.

Contextos:
{contextos_str}

Resumen a evaluar:
{respuesta}

Instrucciones:
1. Extrae las afirmaciones fácticas individuales del Resumen.
2. Si el Resumen declara legítimamente que las referencias no tratan el tema o no resuelven la consulta (abstención), clasifícalo como fiel (score 1.0).
3. Para cada afirmación, determina si está respaldada (true) o no respaldada (false) por el texto de los Contextos.
4. Calcula el score como: (afirmaciones respaldadas) / (total de afirmaciones).

Devuelve ÚNICAMENTE un JSON con este formato:
{{
  "afirmaciones": [
    {{"afirmacion": "...", "respaldada": true, "justificacion": "..."}}
  ],
  "score": 1.0
}}"""
    res = llamar_claude_json(prompt)
    if res is None or 'score' not in res:
        return float('nan')
    return float(res['score'])

def calcular_answer_relevancy(pregunta, respuesta):
    if not respuesta:
        return float('nan')
    prompt = f"""Eres un evaluador riguroso de relevancia de respuestas para sistemas RAG.
Determina qué tan relevante, pertinente y directamente orientada a la pregunta del usuario es la respuesta generada (Resumen).
No evalúes si el artículo es legalmente correcto (eso lo evalúa el contexto), sino si el texto atiende lo preguntado sin divagaciones.

Pregunta del usuario:
{pregunta}

Resumen generado:
{respuesta}

Escala:
- 1.0: Atiende directamente y con claridad la inquietud planteada (o declara honestamente los límites).
- 0.7: Aborda parcialmente la temática con algún aspecto tangencial.
- 0.3 - 0.5: Habla de temas laborales pero lejanos a los hechos del usuario.
- 0.0: Irrelevante, evasivo sin justificación o fuera de tema.

Devuelve ÚNICAMENTE un JSON:
{{
  "justificacion": "...",
  "score": 0.85
}}"""
    res = llamar_claude_json(prompt)
    if res is None or 'score' not in res:
        return float('nan')
    return float(res['score'])

def calcular_context_recall(referencia, contextos):
    if not referencia:
        return float('nan')
    contextos_str = "\n\n---\n\n".join(f"[Articulo {i+1}]: {c}" for i, c in enumerate(contextos))
    prompt = f"""Eres un evaluador legal riguroso de recuperación de información.
Determina si la información y normas esperadas descritas en la 'Referencia de Verdad Terreno' están presentes en los 'Contextos Recuperados'.

Referencia de Verdad Terreno (expected):
{referencia}

Contextos Recuperados:
{contextos_str}

Instrucciones:
1. Descompón la Referencia en sus afirmaciones o requisitos legales clave (norma específica, causal o derecho).
2. Verifica si el texto de los Contextos contiene esa norma o esa información sustancial.
3. Calcula: (afirmaciones encontradas en contexto) / (total afirmaciones de la referencia).
   Si la norma esperada no llegó en los contextos, el score es 0.0.

Devuelve ÚNICAMENTE un JSON:
{{
  "hechos_esperados": [
    {{"hecho": "...", "encontrado_en_contexto": true, "justificacion": "..."}}
  ],
  "score": 1.0
}}"""
    res = llamar_claude_json(prompt)
    if res is None or 'score' not in res:
        return float('nan')
    return float(res['score'])

def calcular_context_precision(pregunta, referencia, contextos):
    if not contextos:
        return float('nan')
    contextos_str = "\n\n---\n\n".join(f"[Articulo {i+1} (Posición {i+1})]: {c}" for i, c in enumerate(contextos))
    prompt = f"""Eres un evaluador legal riguroso de precisión de recuperación para RAG.
Evalúa la relevancia de cada uno de los artículos recuperados (en su orden de ranking: Posición 1, 2, 3) respecto a la consulta y la referencia esperada.

Pregunta del usuario:
{pregunta}

Referencia esperada (expected):
{referencia}

Artículos recuperados en orden:
{contextos_str}

Instrucciones:
1. Para cada artículo (1, 2, 3), indica si es relevante (true) o irrelevante/ruido (false) para resolver la consulta según la referencia.
2. Calcula Context Precision usando Average Precision (AP@k):
   Precisión@k = (relevantes hasta posición k) / k
   Score = Suma(Precisión@k * es_relevante_k) / (Total de relevantes en top 3)
   Si ningún artículo es relevante, el score es 0.0.

Devuelve ÚNICAMENTE un JSON:
{{
  "evaluacion_posiciones": [
    {{"posicion": 1, "relevante": true, "justificacion": "..."}}
  ],
  "score": 1.0
}}"""
    res = llamar_claude_json(prompt)
    if res is None or 'score' not in res:
        return float('nan')
    return float(res['score'])

### 5. Verificación de comportamiento

Antes de correr los 13 casos se prueban a mano tres situaciones conocidas:
1. M2-13 en el avanzado, donde sí llega la norma esperada: el `context_recall` debe ser alto.
2. M2-02 en el ingenuo, donde la norma esperada no llega: debe ser bajo o cero.
3. M2-12 (fuera de dominio) en el avanzado: muestra cómo puntúa el juez una abstención.

In [5]:
row_m2_13 = pd.read_csv(RAIZ / 'notebooks/rag/resultados/avanzado/scorecard_rag_avanzado_detalle.csv').query('id == "M2-13"').iloc[0]
ctx_13 = obtener_contextos(row_m2_13['articulos_recuperados'])
cr_13 = calcular_context_recall(eval_map['M2-13']['expected'], ctx_13)
print(f"Sanity Check 1 (M2-13 con norma correcta esperada): Context Recall = {cr_13:.2f} (Esperado: Alto)")

row_m2_02 = pd.read_csv(RAIZ / 'notebooks/rag/resultados/ingenuo/scorecard_rag_ingenuo_detalle.csv').query('id == "M2-02"').iloc[0]
ctx_02 = obtener_contextos(row_m2_02['articulos_recuperados'])
cr_02 = calcular_context_recall(eval_map['M2-02']['expected'], ctx_02)
print(f"Sanity Check 2 (M2-02 con norma ausente): Context Recall = {cr_02:.2f} (Esperado: 0.00)")

row_m2_12 = pd.read_csv(RAIZ / 'notebooks/rag/resultados/avanzado/scorecard_rag_avanzado_detalle.csv').query('id == "M2-12"').iloc[0]
res_12 = extraer_resumen(row_m2_12['respuesta_sistema'])
ctx_12 = obtener_contextos(row_m2_12['articulos_recuperados'])
f_12 = calcular_faithfulness(res_12, ctx_12)
print(f"Sanity Check 3 (M2-12 abstención fuera de dominio): Faithfulness = {f_12:.2f}")

Sanity Check 1 (M2-13 con norma correcta esperada): Context Recall = 0.67 (Esperado: Alto)
Sanity Check 2 (M2-02 con norma ausente): Context Recall = 0.00 (Esperado: 0.00)
Sanity Check 3 (M2-12 abstención fuera de dominio): Faithfulness = 0.00


In [6]:

def evaluar_sistema_ragas(nombre_sistema, ruta_scorecard_detalle, verbose=True):
    df_detalle = pd.read_csv(ruta_scorecard_detalle)
    if verbose:
        print(f"\n--- Evaluando {nombre_sistema} ({len(df_detalle)} casos) ---")
    filas = []
    for idx, row in df_detalle.iterrows():
        cid = row['id']
        case_type = row.get('case_type', '')
        pregunta = row['input']
        respuesta = extraer_resumen(row['respuesta_sistema'])
        citas_str = row['articulos_recuperados']
        contextos = obtener_contextos(citas_str)
        referencia = eval_map.get(cid, {}).get('expected', '')

        f_score = calcular_faithfulness(respuesta, contextos)
        ar_score = calcular_answer_relevancy(pregunta, respuesta)
        cr_score = calcular_context_recall(referencia, contextos)
        cp_score = calcular_context_precision(pregunta, referencia, contextos)

        if verbose:
            def _fmt(v): return f"{v:.2f}" if v == v else " NaN"  # NaN != NaN
            print(f"[{idx+1:02d}/13] {cid} ({case_type:<15}) | CP: {_fmt(cp_score)} | CR: {_fmt(cr_score)} | Faith: {_fmt(f_score)} | Rel: {_fmt(ar_score)}")

        filas.append({
            'sistema': nombre_sistema,
            'id': cid,
            'case_type': case_type,
            'pregunta': pregunta,
            'articulos_recuperados': citas_str,
            'context_precision': round(cp_score, 4),
            'context_recall': round(cr_score, 4),
            'faithfulness': round(f_score, 4),
            'answer_relevancy': round(ar_score, 4)
        })
    if verbose:
        import math
        fallos = {
            'context_precision': sum(1 for f in filas if math.isnan(f['context_precision'])),
            'context_recall': sum(1 for f in filas if math.isnan(f['context_recall'])),
            'faithfulness': sum(1 for f in filas if math.isnan(f['faithfulness'])),
            'answer_relevancy': sum(1 for f in filas if math.isnan(f['answer_relevancy'])),
        }
        total_fallos = sum(fallos.values())
        if total_fallos > 0:
            print(f"[FALLOS API] {nombre_sistema}: {fallos}")
        else:
            print(f"[OK] Sin fallos de API en {nombre_sistema}.")
    return pd.DataFrame(filas)

In [7]:
RUTA_CSV_RAGAS = RAIZ / 'notebooks/rag/resultados/ragas/ragas_m3.csv'

RECALCULAR = True

if RUTA_CSV_RAGAS.is_file() and not RECALCULAR:
    print(f'Cargando resultados preexistentes desde: {RUTA_CSV_RAGAS}')
    df_final = pd.read_csv(RUTA_CSV_RAGAS)
else:
    print('Calculando métricas RAGAS sobre RAG Ingenuo y RAG Avanzado...')
    df_ing = evaluar_sistema_ragas('rag_ingenuo', RAIZ / 'notebooks/rag/resultados/ingenuo/scorecard_rag_ingenuo_detalle.csv')
    df_adv = evaluar_sistema_ragas('rag_avanzado', RAIZ / 'notebooks/rag/resultados/avanzado/scorecard_rag_avanzado_detalle.csv')
    df_base = pd.concat([df_ing, df_adv], ignore_index=True)

    promedios = []
    for sist in ['rag_ingenuo', 'rag_avanzado']:
        sub = df_base[df_base['sistema'] == sist]
        promedios.append({
            'sistema': sist,
            'id': 'PROMEDIO',
            'case_type': 'promedio',
            'pregunta': 'Promedio general del sistema',
            'articulos_recuperados': '-',
            'context_precision': round(sub['context_precision'].dropna().mean(), 4),
            'context_recall': round(sub['context_recall'].dropna().mean(), 4),
            'faithfulness': round(sub['faithfulness'].dropna().mean(), 4),
            'answer_relevancy': round(sub['answer_relevancy'].dropna().mean(), 4)
        })
    df_final = pd.concat([df_base, pd.DataFrame(promedios)], ignore_index=True)
    RUTA_CSV_RAGAS.parent.mkdir(parents=True, exist_ok=True)
    df_final.to_csv(RUTA_CSV_RAGAS, index=False, encoding='utf-8-sig')
    print(f'Resultados exportados a: {RUTA_CSV_RAGAS}')

tabla_promedios = df_final[df_final['id'] == 'PROMEDIO'][['sistema', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']]
display(tabla_promedios)

Calculando métricas RAGAS sobre RAG Ingenuo y RAG Avanzado...

--- Evaluando rag_ingenuo (13 casos) ---
[01/13] M2-01 (estandar       ) | CP: 0.83 | CR: 1.00 | Faith: 0.67 | Rel: 0.75
[02/13] M2-02 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 0.00 | Rel: 0.00
[03/13] M2-03 (estandar       ) | CP: 0.00 | CR: 0.25 | Faith: 0.67 | Rel: 0.35
[04/13] M2-04 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 0.83 | Rel: 0.35
[05/13] M2-05 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 1.00 | Rel: 0.45
[06/13] M2-06 (estandar       ) | CP: 0.33 | CR: 0.67 | Faith: 0.75 | Rel: 0.40
[07/13] M2-07 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 0.50 | Rel: 0.55
[08/13] M2-08 (estandar       ) | CP: 0.00 | CR: 0.25 | Faith: 1.00 | Rel: 0.25
[09/13] M2-09 (dificil        ) | CP: 0.33 | CR: 0.17 | Faith: 0.57 | Rel: 0.72
[10/13] M2-10 (dificil        ) | CP: 0.83 | CR: 0.75 | Faith: 1.00 | Rel: 0.35
[11/13] M2-11 (premisa_falsa  ) | CP: 0.17 | CR: 0.00 | Faith: 0.40 | Rel: 0.25
[12/13] M2-12 (f

,sistema,context_precision,context_recall,faithfulness,answer_relevancy
26,rag_ingenuo,0.1915,0.2531,0.6531,0.3554
27,rag_avanzado,0.1599,0.2577,0.7869,0.3923


In [8]:
RUTA_FINAL_DETALLE = RAIZ / 'notebooks/rag/resultados/final/scorecard_sistema_final_detalle.csv'

if RUTA_FINAL_DETALLE.is_file():
    print(f'Archivo del Sistema Final encontrado: {RUTA_FINAL_DETALLE}')
    df_fin = evaluar_sistema_ragas('sistema_final', RUTA_FINAL_DETALLE)
    prom_fin = pd.DataFrame([{
        'sistema': 'sistema_final',
        'id': 'PROMEDIO',
        'case_type': 'promedio',
        'pregunta': 'Promedio general del sistema',
        'articulos_recuperados': '-',
        'context_precision': round(df_fin['context_precision'].mean(), 4),
        'context_recall': round(df_fin['context_recall'].mean(), 4),
        'faithfulness': round(df_fin['faithfulness'].mean(), 4),
        'answer_relevancy': round(df_fin['answer_relevancy'].mean(), 4)
    }])
    df_actualizado = pd.concat([df_final[df_final['sistema'] != 'sistema_final'], df_fin, prom_fin], ignore_index=True)
    df_actualizado.to_csv(RUTA_CSV_RAGAS, index=False, encoding='utf-8-sig')
    print('Sistema Final integrado exitosamente a ragas_m3.csv')
    display(df_actualizado[df_actualizado['id'] == 'PROMEDIO'][['sistema', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']])
else:
    print('No se encontró resultados/final/scorecard_sistema_final_detalle.csv; se omite el sistema final.')

Archivo del Sistema Final encontrado: /content/cabrejos-ortiz-valencia-lopez/notebooks/rag/resultados/final/scorecard_sistema_final_detalle.csv

--- Evaluando sistema_final (13 casos) ---
[01/13] M2-01 (estandar       ) | CP: 0.33 | CR: 1.00 | Faith: 0.67 | Rel: 0.65
[02/13] M2-02 (estandar       ) | CP:  NaN | CR: 0.00 | Faith: 0.00 | Rel: 0.75
[03/13] M2-03 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 1.00 | Rel: 0.35
[04/13] M2-04 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 1.00 | Rel: 0.25
[05/13] M2-05 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 0.80 | Rel: 0.35
[06/13] M2-06 (estandar       ) | CP: 0.33 | CR: 0.50 | Faith: 1.00 | Rel: 0.40
[07/13] M2-07 (estandar       ) | CP:  NaN | CR: 0.00 | Faith: 0.00 | Rel: 0.75
[08/13] M2-08 (estandar       ) | CP: 0.00 | CR: 0.00 | Faith: 0.25 | Rel: 0.15
[09/13] M2-09 (dificil        ) | CP: 0.83 | CR: 0.67 | Faith: 0.89 | Rel: 0.35
[10/13] M2-10 (dificil        ) | CP: 0.00 | CR: 0.00 | Faith: 0.75 | Rel: 0.35
[11/13] M2-1

,sistema,context_precision,context_recall,faithfulness,answer_relevancy
26,rag_ingenuo,0.1915,0.2531,0.6531,0.3554
27,rag_avanzado,0.1599,0.2577,0.7869,0.3923
41,sistema_final,0.1660,0.2246,0.6431,0.4577


### 9. Lectura de los resultados

Las cifras están en la tabla de promedios de arriba y en `resultados/ragas/ragas_m3.csv`. El juez es un LLM no determinista: con las mismas entradas, el `context_recall` del ingenuo dio 0,19, 0,28, 0,24 y 0,25 en distintas corridas. Por eso solo se interpretan diferencias claras, no las de ~0,1.

**Recuperación frente a generación.** El `context_recall` es bajo en los tres sistemas. En M2-03, M2-04, M2-07 y M2-08 ningún sistema recupera la norma esperada, así que el decoder resume artículos que no contienen la respuesta. Esto apunta a la recuperación como falla principal, más que al generador. La `faithfulness` es moderada o alta: Qwen tiende a resumir lo que recibe y rara vez cita normas ajenas. Las diferencias entre el ingenuo y el avanzado en las cuatro métricas están dentro del ruido del juez, por lo que no se concluye que el avanzado genere mejor ni peor.

**`recall_3` del harness frente a `context_recall` de RAGAS.** El `recall_3` exige la cita exacta (p. ej. `CST Art. 239`). En M2-01 el sistema trae la `Ley 1822 de 2017 Art. 2`, que reproduce esa regla, y el harness da 0,0; RAGAS lo evalúa por contenido y le asigna un recall alto. Las dos métricas miden cosas distintas y se complementan.

**Casos especiales.** En M2-12 (fuera de dominio) el avanzado se abstiene, pero el juez le da `faithfulness` 0,0 aunque el prompt indica que una abstención es fiel: una limitación del juez, no del decoder. En M2-11 (premisa falsa) el resumen no desmiente la premisa y el juez lo califica como relevante: la válvula de escape depende del decoder y no es una conducta controlada del sistema.
**Sistema final (agente).** En M2-02 y M2-07 el agente eligió `no_tool` y no hubo artículos: `context_precision` queda en NaN y se excluye del promedio (el mensaje «FALLOS API: context_precision 2» de la celda anterior corresponde a estos dos casos, no a fallos de la API), mientras que `faithfulness` sale 0,0 y `answer_relevancy` 0,75 porque el juez puntúa un texto fijo contra un contexto vacío. Esos dos valores no miden calidad de generación y distorsionan el promedio de 13 casos del sistema final. Para compararlo con justicia se promedian los 11 casos donde los tres sistemas recuperaron artículos: context precision 0,23 / 0,19 / 0,17, context recall 0,30 / 0,29 / 0,27, faithfulness 0,73 / 0,77 / 0,76 y answer relevancy 0,37 / 0,37 / 0,40 (ingenuo / avanzado / final). Las diferencias están dentro del ruido del juez y no permiten ordenar los sistemas.